# Chapitre 5 - Cours 1 : polynômes par morceaux, splines et B-splines

Ce premier notebook du chapitre 5 (*Basis Expansions and Regularization*) présente
l'idée centrale du chapitre : **remplacer la variable d'entrée $X$ par des
transformations $h_1(X), \dots, h_M(X)$** puis ajuster un modèle *linéaire* dans ces
nouvelles variables. Nous verrons :

1. les expansions en bases de fonctions (§5.1) ;
2. les polynômes par morceaux et les splines (§5.2, figures 5.1 et 5.2) ;
3. la base des B-splines et la récurrence de Cox-de Boor (annexe du chapitre, figure 5.20) ;
4. les splines cubiques naturelles et la variance ponctuelle (§5.2.1, figure 5.3) ;
5. l'exemple de la reconnaissance de phonèmes : filtrer les entrées par une base de
   splines (§5.2.3 et §5.3, figure 5.5).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import (piecewise_polynomial_basis, truncated_power_basis,
                              bspline_basis, natural_spline_basis, logistic_irls)
set_style()

## 1. Expansions en bases de fonctions

Les modèles linéaires des chapitres 3 et 4 supposent que la fonction de régression
$f(X) = \mathbb E(Y \mid X)$ est linéaire en $X$. C'est rarement vrai, mais c'est une
approximation commode. Pour aller au-delà **sans abandonner les outils linéaires**, on
se donne des fonctions $h_m : \mathbb R^p \to \mathbb R$, $m = 1, \dots, M$, et l'on pose
(éq. (5.1) du livre)

$$
f(X) = \sum_{m=1}^{M} \beta_m\, h_m(X).
$$

Le modèle est **linéaire en les paramètres** $\beta_m$ : une fois les $h_m$ choisies,
l'ajustement par moindres carrés est exactement celui du chapitre 3, avec la matrice de
régression $\mathbf H = (h_m(x_i))_{i,m}$ de taille $N \times M$ :
$\hat\beta = (\mathbf H^T \mathbf H)^{-1}\mathbf H^T \mathbf y$.

Exemples de choix de $h_m$ :

- $h_m(X) = X_m$ : on retrouve le modèle linéaire ;
- $h_m(X) = X_j^2$ ou $X_j X_k$ : polynôme de degré 2 (le nombre de termes explose avec $p$) ;
- $h_m(X) = \log X_j$, $\sqrt{X_j}$ : transformations non linéaires d'une seule variable ;
- $h_m(X) = \mathbf 1(L_m \le X_k < U_m)$ : indicatrices d'intervalles, qui donnent des
  fonctions constantes par morceaux.

> **Intuition.** Un polynôme global est trop rigide : modifier les données à un endroit
> modifie l'ajustement partout. Les polynômes **par morceaux** et les splines sont
> *locaux*, ce qui les rend beaucoup plus souples.

Jusqu'au §5.7, $X$ est **unidimensionnel**.

## 2. Polynômes par morceaux

On découpe le domaine de $X$ en intervalles contigus délimités par des **nœuds**
$\xi_1 < \xi_2 < \dots < \xi_K$, et l'on représente $f$ par un polynôme différent sur
chaque intervalle. Nous utilisons des données simulées, avec deux nœuds
$\xi_1 = 1/3$ et $\xi_2 = 2/3$, pour reproduire l'esprit des figures 5.1 et 5.2.

In [2]:
rng = np.random.RandomState(0)
N = 60
x = np.sort(rng.uniform(0, 1, N))
f_true = lambda t: np.sin(2.5 * np.pi * t) * np.exp(-t) + 0.5 * t
y = f_true(x) + 0.25 * rng.randn(N)
knots = np.array([1 / 3, 2 / 3])
grid = np.linspace(0, 1, 400)

### Constante par morceaux

Avec trois fonctions de base
$h_1(X) = \mathbf 1(X < \xi_1)$, $h_2(X) = \mathbf 1(\xi_1 \le X < \xi_2)$,
$h_3(X) = \mathbf 1(\xi_2 \le X)$, les colonnes de $\mathbf H$ ont des supports disjoints,
donc $\mathbf H^T\mathbf H = \mathrm{diag}(N_1, N_2, N_3)$ où $N_m$ est le nombre
d'observations dans la région $m$, et $(\mathbf H^T \mathbf y)_m = \sum_{x_i \in R_m} y_i$.
Ainsi

$$
\hat\beta_m = \frac{1}{N_m}\sum_{x_i \in R_m} y_i = \bar y_m ,
$$

la **moyenne des $y_i$ dans la région $m$**.

In [3]:
def fit_predict(Hfun, x, y, grid):
    """Moindres carrés dans la base Hfun ; retourne l'ajustement sur la grille."""
    H = Hfun(x)
    beta, *_ = np.linalg.lstsq(H, y, rcond=None)
    return Hfun(grid) @ beta, beta

H0 = lambda t: np.column_stack([t < knots[0], (t >= knots[0]) & (t < knots[1]),
                                t >= knots[1]]).astype(float)
_, beta0 = fit_predict(H0, x, y, grid)
moyennes = [y[x < knots[0]].mean(), y[(x >= knots[0]) & (x < knots[1])].mean(),
            y[x >= knots[1]].mean()]
print("coefficients :", np.round(beta0, 4))
print("moyennes     :", np.round(moyennes, 4))

coefficients : [ 0.5832 -0.0866  0.5596]
moyennes     : [ 0.5832 -0.0866  0.5596]


### Linéaire par morceaux, puis continue

Pour un ajustement linéaire par morceaux, on ajoute $h_{m+3}(X) = h_m(X)\,X$ : six
paramètres. En général on préfère une fonction **continue** aux nœuds. La continuité en
$\xi_1$, $f(\xi_1^-) = f(\xi_1^+)$, s'écrit $\beta_1 + \xi_1\beta_4 = \beta_2 + \xi_1\beta_5$ :
c'est une **contrainte linéaire**. Deux nœuds donnent deux contraintes, d'où
$6 - 2 = 4$ paramètres libres.

Une base qui incorpore directement les contraintes est

$$
h_1(X) = 1,\quad h_2(X) = X,\quad h_3(X) = (X - \xi_1)_+,\quad h_4(X) = (X-\xi_2)_+,
$$

où $t_+ = \max(t, 0)$ désigne la partie positive. Chaque $h_m$ est continue et affine
par morceaux ; toute combinaison l'est donc aussi. Réciproquement, une fonction affine par
morceaux continue se décrit par la pente et l'ordonnée du premier morceau, puis par les
**changements de pente** en $\xi_1$ et $\xi_2$ : exactement 4 nombres.

In [4]:
H1 = lambda t: piecewise_polynomial_basis(t, knots, degree=1, continuity=-1)
H1c = lambda t: piecewise_polynomial_basis(t, knots, degree=1, continuity=0)
fits = {
    "Constante par morceaux": fit_predict(H0, x, y, grid)[0],
    "Linéaire par morceaux": fit_predict(H1, x, y, grid)[0],
    "Linéaire par morceaux continue": fit_predict(H1c, x, y, grid)[0],
}

La figure suivante reproduit la figure 5.1 du livre : les trois ajustements, puis (en bas
à droite) la fonction de base $(X - \xi_1)_+$.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
for ax, (titre, fit) in zip(axes.ravel()[:3], fits.items()):
    ax.scatter(x, y, s=14, color=GRAY, alpha=0.8, label="données")
    ax.plot(grid, f_true(grid), color=BLUE, lw=1.5, ls="--", label="vraie $f$")
    for g in np.split(np.arange(len(grid)), np.searchsorted(grid, knots)):
        ax.plot(grid[g], fit[g], color=ORANGE)
    ax.set_title(titre)
    ax.set_ylabel("$y$")
    for k in knots:
        ax.axvline(k, color=GRAY, lw=0.8, ls=":")
ax = axes[1, 1]
ax.plot(grid, np.maximum(grid - knots[0], 0), color=ORANGE)
ax.axvline(knots[0], color=GRAY, lw=0.8, ls=":")
ax.set_title(r"Fonction de base $(X-\xi_1)_+$")
axes[0, 0].legend(loc="lower left")
for ax in axes[1]:
    ax.set_xlabel("$X$")
plt.tight_layout()
plt.show()